In [1]:
import requests
import os
from dotenv import load_dotenv

# Cargar API key desde archivo .env
load_dotenv()
API_KEY = os.getenv("USDA_API_KEY")
BASE_URL = "https://api.nal.usda.gov/fdc/v1"

# Verificar conexión
response = requests.get(
    f"{BASE_URL}/foods/search",
    params={
        "api_key": API_KEY,
        "query": "breakfast cereal",
        "dataType": ["Branded"],
        "pageSize": 5
    }
)
if response.status_code == 200:
    data = response.json()
    print(f"✓ Conexión exitosa")
    print(f"Total productos: {data['totalHits']}")
    for food in data['foods']:
        print(f"  - {food['description']}")
else:
    print(f"✗ Error: {response.status_code}")

✓ Conexión exitosa
Total productos: 14739
  - Bluey Breakfast Cereal
  - BREAKFAST PACK CEREAL
  - Cheerios Breakfast Cereal
  - Cheerios Breakfast Cereal
  - Total Breakfast Cereal


In [2]:
import pandas as pd
import time

def extraer_cereales(api_key, max_productos=15000):
    """Extrae todos los cereales de USDA"""
    todos_los_productos = []
    page = 1
    page_size = 200  # Máximo permitido por la API
    
    while len(todos_los_productos) < max_productos:
        response = requests.get(
            f"{BASE_URL}/foods/search",
            params={
                "api_key": api_key,
                "query": "cereal",
                "dataType": ["Branded"],
                "pageSize": page_size,
                "pageNumber": page
            }
        )
        
        if response.status_code != 200:
            print(f"Error en página {page}: {response.status_code}")
            break
            
        data = response.json()
        productos = data.get('foods', [])
        
        if not productos:
            break
            
        todos_los_productos.extend(productos)
        print(f"Página {page}: {len(productos)} productos (Total: {len(todos_los_productos)})")
        
        page += 1
        time.sleep(0.5)  # Pausa para no saturar la API
    
    return todos_los_productos

# Ejecutar extracción
print("Iniciando extracción...")
productos = extraer_cereales(API_KEY)
print(f"\nTotal extraído: {len(productos)} productos")

Iniciando extracción...
Página 1: 200 productos (Total: 200)
Página 2: 200 productos (Total: 400)
Página 3: 200 productos (Total: 600)
Página 4: 200 productos (Total: 800)
Página 5: 200 productos (Total: 1000)
Página 6: 200 productos (Total: 1200)
Página 7: 200 productos (Total: 1400)
Página 8: 200 productos (Total: 1600)
Página 9: 200 productos (Total: 1800)
Página 10: 200 productos (Total: 2000)
Página 11: 200 productos (Total: 2200)
Página 12: 200 productos (Total: 2400)
Página 13: 200 productos (Total: 2600)
Página 14: 200 productos (Total: 2800)
Página 15: 200 productos (Total: 3000)
Página 16: 200 productos (Total: 3200)
Página 17: 200 productos (Total: 3400)
Página 18: 200 productos (Total: 3600)
Página 19: 200 productos (Total: 3800)
Página 20: 200 productos (Total: 4000)
Página 21: 200 productos (Total: 4200)
Página 22: 200 productos (Total: 4400)
Página 23: 200 productos (Total: 4600)
Página 24: 200 productos (Total: 4800)
Página 25: 200 productos (Total: 5000)
Página 26: 200

In [3]:
# Extraer campos relevantes
datos = []
for p in productos:
    datos.append({
        'fdc_id': p.get('fdcId'),
        'description': p.get('description'),
        'brand_owner': p.get('brandOwner'),
        'serving_size': p.get('servingSize'),
        'serving_size_unit': p.get('servingSizeUnit'),
        'food_category': p.get('foodCategory'),
        # Nutrientes
        **{n['nutrientName']: n.get('value') for n in p.get('foodNutrients', [])}
    })

# Crear DataFrame y guardar
df = pd.DataFrame(datos)
df.to_csv('/Users/aracelli/Documents/GitHub/TFM-similitud-nutricional/data/raw/cereales_raw.csv', index=False)

print(f"Guardado: {len(df)} productos")
print(f"Columnas: {len(df.columns)}")
print(df.head())

Guardado: 11995 productos
Columnas: 81
    fdc_id description                     brand_owner  serving_size  \
0  2642006      CEREAL  President Baking Company, Inc.          37.0   
1  2107531      CEREAL            The Kellogg Company           29.0   
2  1952060      CEREAL       Post Consumer Brands, LLC          28.0   
3  1889809      CEREAL       Post Consumer Brands, LLC          30.0   
4  2140770      CEREAL            The Kellogg Company           53.0   

  serving_size_unit food_category  Protein  Total lipid (fat)  \
0               GRM        Cereal     5.41               0.00   
1                 g        Cereal     6.90               0.00   
2                 g        Cereal    10.70               5.36   
3                 g        Cereal     3.33               3.33   
4                 g        Cereal     5.66               1.89   

   Carbohydrate, by difference  Energy  ...  Valine  Arginine  Histidine  \
0                         89.2   351.0  ...     NaN       NaN

In [13]:
# Verificación
print(f"✓ Extracción completada")
print(f"  Productos: {len(df)}")
print(f"  Columnas: {len(df.columns)}")
print(f"  Archivo: data/raw/cereales_raw.csv")
print(f"  Fecha de extracción: {pd.Timestamp.now().strftime('%Y-%m-%d')}")

✓ Extracción completada
  Productos: 11995
  Columnas: 81
  Archivo: data/raw/cereales_raw.csv
  Fecha de extracción: 2026-05-30
